# Deteccao de Buracos com CNN Baseline

Notebook profissional em portugues para treino e avaliacao de uma CNN baseline usando o dataset local `datasets/whole-detection/archive`.

## 1. Objetivo

- Carregar o dataset local de imagens de pista normal e com buracos.
- Treinar uma CNN baseline para classificacao binaria.
- Avaliar com acuracia, precision, recall, f1-score e matriz de confusao.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

# Localiza a raiz do projeto para importar o pacote src de forma robusta no VS Code.
raiz_atual = Path.cwd().resolve()
projeto_raiz = next((p for p in [raiz_atual, *raiz_atual.parents] if (p / "src").exists()), raiz_atual)

if str(projeto_raiz) not in sys.path:
    sys.path.insert(0, str(projeto_raiz))

from src.deteccao_buracos import (
    MAPA_CLASSES,
    PROPORCAO_VALIDACAO_PADRAO,
    SEMENTE_PADRAO,
    TAMANHO_IMAGEM_PADRAO,
    avaliar_modelo,
    carregar_imagens_rotuladas,
    construir_cnn_baseline,
    obter_caminhos_dataset,
    separar_treino_validacao,
    treinar_modelo,
)

sns.set_theme(style="whitegrid")
print(f"Raiz do projeto: {projeto_raiz}")

## 2. Configuracao

Definimos parametros de reproducibilidade, resolucao de imagem e hiperparametros de treino.

In [ ]:
epocas = 10
batch_size = 32
forma_entrada = (TAMANHO_IMAGEM_PADRAO[0], TAMANHO_IMAGEM_PADRAO[1], 3)

caminhos = obter_caminhos_dataset()
print("Caminhos do dataset:")
for chave, valor in caminhos.items():
    print(f"- {chave}: {valor}")

print(f"\nMapa de classes: {MAPA_CLASSES}")
print(f"Tamanho de imagem: {TAMANHO_IMAGEM_PADRAO}")
print(f"Semente: {SEMENTE_PADRAO}")

## 3. Carregamento e pre-processamento

As imagens sao carregadas ja normalizadas no intervalo `[0, 1]` e com rotulos binarios.

In [ ]:
X, y = carregar_imagens_rotuladas(caminhos["base"], TAMANHO_IMAGEM_PADRAO)

print(f"Shape de X: {X.shape}")
print(f"Shape de y: {y.shape}")
print(f"Faixa de valores em X: min={X.min():.4f}, max={X.max():.4f}")

classes_unicas, contagens = np.unique(y, return_counts=True)
distribuicao = dict(zip(classes_unicas, contagens))
print(f"Distribuicao de classes: {distribuicao}")

## 4. EDA basica

Visualizamos distribuicao de classes e amostras de cada categoria.

In [ ]:
labels_legiveis = ["normal", "potholes"]
valores = [int(np.sum(y == MAPA_CLASSES["normal"])), int(np.sum(y == MAPA_CLASSES["potholes"]))]

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].pie(valores, labels=labels_legiveis, autopct="%1.1f%%", colors=["#2e8b57", "#cc3333"])
axes[0].set_title("Distribuicao de classes")

sns.barplot(x=labels_legiveis, y=valores, ax=axes[1], palette=["#2e8b57", "#cc3333"])
axes[1].set_title("Contagem por classe")
axes[1].set_ylabel("Quantidade")
plt.tight_layout()
plt.show()

# Exibe 4 amostras de cada classe para inspecao visual rapida.
fig, axes = plt.subplots(2, 4, figsize=(12, 6))
indices_normal = np.where(y == MAPA_CLASSES["normal"])[0][:4]
indices_potholes = np.where(y == MAPA_CLASSES["potholes"])[0][:4]

for i, idx in enumerate(indices_normal):
    axes[0, i].imshow(X[idx])
    axes[0, i].set_title("normal")
    axes[0, i].axis("off")

for i, idx in enumerate(indices_potholes):
    axes[1, i].imshow(X[idx])
    axes[1, i].set_title("potholes")
    axes[1, i].axis("off")

plt.tight_layout()
plt.show()

## 5. Separacao treino-validacao

Separacao estratificada para manter proporcao de classes nos dois conjuntos.

In [ ]:
X_treino, X_valid, y_treino, y_valid = separar_treino_validacao(
    X, y, PROPORCAO_VALIDACAO_PADRAO, SEMENTE_PADRAO
)

print(f"X_treino: {X_treino.shape} | y_treino: {y_treino.shape}")
print(f"X_valid: {X_valid.shape} | y_valid: {y_valid.shape}")

## 6. Treino da CNN baseline

Modelo convolucional simples para classificacao binaria.

In [ ]:
modelo = construir_cnn_baseline(forma_entrada)
modelo.summary()

In [ ]:
historico = treinar_modelo(
    modelo=modelo,
    X_treino=X_treino,
    y_treino=y_treino,
    X_valid=X_valid,
    y_valid=y_valid,
    epocas=epocas,
    batch_size=batch_size,
)

## 7. Avaliacao

Calculamos metricas e exibimos a matriz de confusao.

In [ ]:
resultados = avaliar_modelo(modelo, X_valid, y_valid)

print(f"Acuracia de validacao: {resultados['acuracia']:.4f}")
print("\nMatriz de confusao:")
print(resultados["matriz_confusao"])

print("\nRelatorio de classificacao (resumo):")
for classe in ["normal", "potholes"]:
    metricas = resultados["relatorio_classificacao"][classe]
    print(
        f"{classe:9s} | precision={metricas['precision']:.4f} "
        f"recall={metricas['recall']:.4f} f1-score={metricas['f1-score']:.4f}"
    )

In [ ]:
plt.figure(figsize=(6, 5))
sns.heatmap(
    resultados["matriz_confusao"],
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=["normal", "potholes"],
    yticklabels=["normal", "potholes"],
)
plt.title("Matriz de confusao - CNN baseline")
plt.xlabel("Predito")
plt.ylabel("Real")
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(historico.history["loss"], label="Treino")
axes[0].plot(historico.history["val_loss"], label="Validacao")
axes[0].set_title("Loss por epoca")
axes[0].set_xlabel("Epoca")
axes[0].set_ylabel("Loss")
axes[0].legend()

axes[1].plot(historico.history["accuracy"], label="Treino")
axes[1].plot(historico.history["val_accuracy"], label="Validacao")
axes[1].set_title("Acuracia por epoca")
axes[1].set_xlabel("Epoca")
axes[1].set_ylabel("Acuracia")
axes[1].legend()

plt.tight_layout()
plt.show()

## 8. Conclusao

Pipeline baseline concluido com sucesso, usando codigo reutilizavel em `src` e dataset local, pronto para evolucao futura (aumento de dados, tuning e novos modelos).